# CrewAI Job Application Crew

**Project:** Multi-Agent Job Application Pipeline  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent crew** that automates the entire job application workflow — from analyzing a job description to preparing for the interview.

### The Agent Roster

| # | Agent | Role | What It Produces |
|---|-------|------|-----------------|
| 1 | **JD Analyzer** | Requirements Extractor | Structured breakdown of what the employer actually wants |
| 2 | **Resume Tailor** | Resume Optimizer | Rewritten resume bullets aligned to the JD's priorities |
| 3 | **Cover Letter Writer** | Persuasion Specialist | Targeted cover letter connecting experience to requirements |
| 4 | **Interview Coach** | Prep Strategist | Predicted questions, STAR-format answers, and talking points |

### Pipeline Architecture

```
Job Description + Raw Resume
         │
         ▼
  ┌──────────────┐
  │  JD Analyzer │  Extracts: must-haves, nice-to-haves, culture signals, hidden requirements
  └──────┬───────┘
         │ Structured JD analysis
         ▼
  ┌──────────────┐
  │ Resume Tailor│  Rewrites bullets, reorders sections, adds keyword matches
  └──────┬───────┘
         │ Tailored resume
         ▼
  ┌─────────────────┐
  │ Cover Letter    │  Writes a targeted cover letter using JD analysis + tailored resume
  │ Writer          │
  └──────┬──────────┘
         │ Cover letter
         ▼
  ┌─────────────────┐
  │ Interview Coach │  Generates questions, STAR answers, and strategy from ALL prior context
  └─────────────────┘
         │
         ▼
  Complete application package + interview prep
```

---

## Learning Notes: Role Design in CrewAI

### What Is Role Design?

**Role design** is the art of defining an agent's identity so the LLM behaves like a domain expert rather than a generic assistant. Each agent has three identity properties:

| Property | What It Does | Analogy |
|----------|-------------|---------|
| `role` | Job title — anchors the agent's professional identity | "You are a ___" |
| `goal` | Mission — drives what the agent prioritizes and optimizes for | "Your objective is to ___" |
| `backstory` | Expertise context — shapes HOW the agent thinks and works | "You bring 15 years of experience in ___" |

### Why Role Design Matters

The same LLM produces **dramatically different outputs** based on role design:

```
Poor role design:    role="AI Assistant"  →  Generic, unfocused output
                     goal="Help the user"
                     backstory="You are helpful"

Strong role design:  role="Senior Technical Recruiter"  →  Specific, expert-level output
                     goal="Identify the top 3 skills gaps between the resume and JD"
                     backstory="You've screened 10,000+ resumes at FAANG companies..."
```

### Five Principles of Effective Role Design

1. **Specificity wins** — "Senior Resume Writer specializing in tech careers" > "Writer"
2. **Backstory shapes reasoning** — Include years of experience, methodologies used, and quality standards
3. **Goal drives output** — A narrowly scoped goal produces better results than "do a good job"
4. **Domain vocabulary matters** — Use industry terms in the backstory ("ATS optimization", "STAR method")
5. **Output format in backstory** — Tell the agent HOW to structure its deliverable

This notebook demonstrates these principles in each agent definition, with annotations explaining each design choice.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

All four agents share the same local Ollama model. Their **different behaviors come entirely from role design** — not from different models or temperatures.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Input Data — Job Description & Resume

We define the raw inputs that the crew will process. In production, these could come from file uploads, web scraping, or an API.

In [ ]:
# =====================================================
# SAMPLE JOB DESCRIPTION
# =====================================================
JOB_DESCRIPTION = """
Senior Machine Learning Engineer — FinTech Startup (Series B)

About Us:
We're building the next generation of fraud detection for digital banking.
Our platform processes 50M+ transactions daily using real-time ML pipelines.

Requirements:
- 5+ years of experience in machine learning engineering
- Strong Python skills (pandas, scikit-learn, PyTorch or TensorFlow)
- Experience with real-time ML serving (MLflow, Seldon, or TorchServe)
- Familiarity with stream processing (Kafka, Flink, or Spark Streaming)
- Experience with cloud infrastructure (AWS or GCP)
- Knowledge of fraud detection or anomaly detection is a strong plus
- Experience with A/B testing and experiment tracking
- Strong SQL skills and experience with data warehouses (Snowflake, BigQuery)

Nice to Have:
- Experience with graph neural networks for fraud detection
- Contributions to open-source ML projects
- Published research in ML/AI
- Experience in regulated industries (finance, healthcare)

What We Offer:
- Competitive salary ($180k-$240k) + equity
- Remote-first culture with quarterly on-sites
- $5k annual learning budget
- Work directly with the founding team

Culture: Fast-paced, ownership-driven, data-obsessed. We ship weekly
and measure everything. You'll own end-to-end ML systems from research
to production.
"""

print("Job Description loaded")
print(f"Length: {len(JOB_DESCRIPTION):,} characters")

In [ ]:
# =====================================================
# SAMPLE RESUME (raw text)
# =====================================================
RESUME = """
ALEX CHEN
Senior Data Scientist | ML Engineer
San Francisco, CA | alex.chen@email.com | github.com/alexchen

EXPERIENCE:

Machine Learning Engineer — DataCorp Inc. (2021-Present)
- Built and deployed 12+ ML models for customer churn prediction and
  recommendation systems using Python, scikit-learn, and TensorFlow
- Designed real-time inference pipeline using FastAPI and Redis,
  serving 10M+ predictions/day with <50ms p99 latency
- Implemented A/B testing framework that increased model iteration
  speed by 3x and improved conversion by 12%
- Migrated ML infrastructure from on-prem to AWS (SageMaker, S3, Lambda)
- Reduced model training costs by 40% through spot instance optimization

Data Scientist — Analytics Corp (2019-2021)
- Developed anomaly detection system for network security using
  Isolation Forest and autoencoders, catching 94% of intrusions
- Built ETL pipelines processing 2TB/day using PySpark and Airflow
- Created dashboards and reporting using SQL (PostgreSQL) and Tableau
- Collaborated with product teams to define metrics and run experiments

ML Research Intern — University AI Lab (2018-2019)
- Published paper on semi-supervised learning for imbalanced datasets
  at AAAI Workshop 2019
- Implemented graph-based anomaly detection using PyTorch Geometric

EDUCATION:
M.S. Computer Science — Stanford University (2019)
B.S. Mathematics — UC Berkeley (2017)

SKILLS:
Python, PyTorch, TensorFlow, scikit-learn, SQL, PySpark, AWS,
Docker, Git, FastAPI, Redis, Airflow, A/B Testing, MLflow
"""

print("Resume loaded")
print(f"Length: {len(RESUME):,} characters")

## 4. Define the Agents

Each agent below includes **role design annotations** explaining why specific design choices were made.

### 4.1 JD Analyzer Agent

**Role Design Notes:**

| Design Choice | Why |
|--------------|-----|
| Role: "Senior Technical Recruiter" | Frames analysis from the HIRING side — what do they actually want? |
| Goal mentions "hidden requirements" | Pushes the agent to read between the lines, not just list bullet points |
| Backstory cites "10,000+ JDs" | Establishes authority; the agent will pattern-match like an expert |
| Backstory includes "ATS systems" | Primes the agent to think about keyword matching and screening criteria |
| Output format specified | "Structured analysis with priority tiers" ensures downstream agents get clean input |

**Anti-pattern avoided:** A generic `role="Job Analyst"` with `backstory="You analyze jobs"` would produce a surface-level summary instead of a strategic breakdown.

In [ ]:
jd_analyzer = Agent(
    role="Senior Technical Recruiter & JD Analyst",
    goal=(
        "Deconstruct the job description to identify explicit requirements, "
        "implicit expectations, hidden priorities, culture signals, and the "
        "specific skills/experiences that will differentiate a top candidate. "
        "Rank requirements by priority: must-have vs nice-to-have vs hidden."
    ),
    backstory=(
        "You are a senior technical recruiter who has analyzed 10,000+ job descriptions "
        "across tech companies from startups to FAANG. You know that job descriptions "
        "are imperfect signals — what's listed first matters most, 'nice to haves' are "
        "often dealbreakers, and the language reveals culture (e.g., 'fast-paced' means "
        "long hours, 'ownership-driven' means small team). You understand ATS keyword "
        "matching and know which skills are truly required vs aspirational. You always "
        "produce a structured analysis with: (1) ranked requirements, (2) keyword map for "
        "ATS optimization, (3) culture/team signals, and (4) red flags or concerns."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {jd_analyzer.role}")

### 4.2 Resume Tailor Agent

**Role Design Notes:**

| Design Choice | Why |
|--------------|-----|
| Role: "ATS-Optimization Specialist" | Not just "resume writer" — emphasizes the technical screening angle |
| Goal mentions "without fabrication" | Critical safety constraint — prevents the LLM from inventing experience |
| Backstory cites "reorder, reframe, requantify" | Three specific techniques the agent will apply |
| Backstory mentions "XYZ formula" | Industry-standard bullet format (did X, measured by Y, resulting in Z) |
| `allow_delegation=False` | The resume tailor must do its own work; delegation would lose nuance |

**Role design principle demonstrated:** Including **methodologies** in the backstory (XYZ formula, ATS keyword density) gives the agent concrete techniques to apply, not just vague instructions.

In [ ]:
resume_tailor = Agent(
    role="Resume Optimization & ATS Specialist",
    goal=(
        "Tailor the resume to maximize match with the job description. Rewrite "
        "bullet points to highlight relevant experience, reorder sections by "
        "relevance, inject ATS-friendly keywords, and quantify achievements — "
        "all without fabricating or exaggerating experience."
    ),
    backstory=(
        "You are a professional resume writer who has optimized 5,000+ resumes for "
        "tech professionals, achieving a 70% interview callback rate. You specialize "
        "in ATS (Applicant Tracking System) optimization — you know that most resumes "
        "are filtered by algorithms before a human ever sees them. Your approach: "
        "(1) Mirror exact keywords from the JD in the resume, (2) Reorder bullet points "
        "so the most relevant experience appears first, (3) Use the XYZ formula for "
        "bullets: 'Accomplished [X] as measured by [Y] by doing [Z]', (4) Add a "
        "tailored summary section matching the role, (5) Flag any gaps where the "
        "candidate's experience doesn't match and suggest how to address them. "
        "CRITICAL: You NEVER invent experience. You reframe existing experience to "
        "highlight relevance, but every claim must be truthful."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {resume_tailor.role}")

### 4.3 Cover Letter Writer Agent

**Role Design Notes:**

| Design Choice | Why |
|--------------|-----|
| Role includes "Storyteller" | Cover letters are narratives, not lists — the role title primes storytelling |
| Goal specifies "three bridges" | Forces the agent to connect experience→requirements→value, not just summarize |
| Backstory mentions "300 words" | Length constraint embedded in the persona, not just the task |
| Backstory cites "Problem-Solution-Impact" | A specific framework the agent will follow |
| Backstory warns "never generic" | Prevents the "I was excited to see your posting" opener |

**Role design principle demonstrated:** Embedding **constraints in the backstory** (word count, no generic openers) is more effective than putting them only in the task description — the agent treats backstory as core identity, not just instructions.

In [ ]:
cover_letter_writer = Agent(
    role="Executive Cover Letter Strategist & Storyteller",
    goal=(
        "Write a compelling, personalized cover letter that builds three bridges: "
        "(1) connects the candidate's specific experience to the role's requirements, "
        "(2) demonstrates genuine understanding of the company's mission, and "
        "(3) articulates unique value the candidate brings beyond the job description."
    ),
    backstory=(
        "You are a career strategist who has written cover letters for executives and "
        "senior engineers at companies like Google, Stripe, and Databricks. Your letters "
        "have a 40% response rate because they follow a proven structure: "
        "(1) HOOK — Open with a specific, relevant accomplishment (never 'I was excited'), "
        "(2) BRIDGE — Connect 2-3 key experiences directly to the role's top priorities, "
        "(3) INSIGHT — Show understanding of the company's challenges or mission, "
        "(4) DIFFERENTIATION — Articulate what makes this candidate uniquely valuable, "
        "(5) CLOSE — Confident, specific CTA (not 'I look forward to hearing from you'). "
        "Your letters are always under 300 words, use the candidate's authentic voice, "
        "and never repeat the resume — they add context, motivation, and narrative that "
        "the resume can't convey. You follow the Problem-Solution-Impact framework."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {cover_letter_writer.role}")

### 4.4 Interview Coach Agent

**Role Design Notes:**

| Design Choice | Why |
|--------------|-----|
| Role: "Senior Interview Coach" | "Coach" implies strategy, not just Q&A — the agent will prepare the candidate |
| Goal mentions "STAR format" | The standard behavioral interview framework — shows domain expertise |
| Backstory cites "both sides of the table" | The agent thinks like an interviewer AND a candidate |
| Backstory includes "red-flag questions" | Goes beyond basic prep — anticipates tricky questions |
| Backstory specifies "company-specific angles" | Personalized prep, not generic interview advice |

**Role design principle demonstrated:** Giving the agent **dual perspectives** ("both sides of the table") produces richer output because the LLM models both the interviewer's evaluation criteria AND the candidate's presentation strategy.

In [ ]:
interview_coach = Agent(
    role="Senior Technical Interview Coach",
    goal=(
        "Prepare the candidate for the interview by predicting likely questions "
        "(technical, behavioral, and culture-fit), crafting STAR-format answers "
        "using the candidate's actual experience, identifying potential weak spots, "
        "and developing a strategy for handling difficult questions."
    ),
    backstory=(
        "You are a senior interview coach who has prepared 2,000+ candidates for "
        "tech interviews at companies from Series A startups to FAANG. You've sat "
        "on both sides of the table — as a hiring manager who has conducted 500+ "
        "interviews and as a coach who has analyzed interview feedback patterns. "
        "You know that interviews test three things: (1) Can you do the job? (technical), "
        "(2) Have you done similar work? (behavioral/STAR), (3) Will you fit the team? "
        "(culture). For each predicted question, you provide: the question, why the "
        "interviewer asks it, a STAR-format answer draft using the candidate's real "
        "experience, and coaching notes on delivery. You also identify 'red-flag' "
        "questions — areas where the candidate's background has gaps — and prepare "
        "honest, strategic responses. You always include company-specific angles "
        "based on the JD's culture signals."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {interview_coach.role}")

### Agent Roster Summary

In [ ]:
agents = [jd_analyzer, resume_tailor, cover_letter_writer, interview_coach]

print("=" * 70)
print("JOB APPLICATION CREW ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Goal: {agent.goal[:90]}...")
    print(f"    Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)

## 5. Define Tasks

### Role Design in Tasks

Task design complements agent role design. The `description` tells the agent WHAT to do; the agent's role/backstory determines HOW it does it.

**Key principle:** Tasks should specify the **deliverable format** via `expected_output`. This ensures each agent's output is structured for the next agent to consume.

### 5.1 Task 1 — JD Analysis

In [ ]:
task_jd_analysis = Task(
    description=(
        f"Analyze this job description in depth:\n\n{JOB_DESCRIPTION}\n\n"
        "Produce a structured analysis covering:\n"
        "1. **Must-Have Requirements**: Skills/experience that are non-negotiable (ranked by importance)\n"
        "2. **Nice-to-Have Requirements**: Differentiators that strengthen the application\n"
        "3. **Hidden Requirements**: Implicit expectations not explicitly stated (read between the lines)\n"
        "4. **ATS Keyword Map**: Exact keywords/phrases to mirror in the resume (with frequency targets)\n"
        "5. **Culture Signals**: What the language reveals about team dynamics, pace, and expectations\n"
        "6. **Seniority Calibration**: What level of experience they truly need vs what they listed\n"
        "7. **Red Flags / Concerns**: Anything a candidate should be cautious about\n"
    ),
    expected_output=(
        "A structured JD analysis with 7 sections: must-haves (ranked), nice-to-haves, "
        "hidden requirements, ATS keyword map, culture signals, seniority calibration, "
        "and red flags. Each section should have specific, actionable items."
    ),
    agent=jd_analyzer,
)

print(f"Task created: JD Analysis -> {task_jd_analysis.agent.role}")

### 5.2 Task 2 — Resume Tailoring

In [ ]:
task_resume_tailor = Task(
    description=(
        f"Tailor this resume to match the analyzed job description:\n\n"
        f"RESUME:\n{RESUME}\n\n"
        "Using the JD analysis from the previous agent, perform:\n"
        "1. **Summary Section**: Write a 2-3 sentence professional summary tailored to this role\n"
        "2. **Bullet Rewriting**: Rewrite each bullet point to:\n"
        "   - Mirror ATS keywords from the JD analysis\n"
        "   - Lead with the most relevant experience for this specific role\n"
        "   - Use XYZ format: 'Accomplished [X] measured by [Y] by doing [Z]'\n"
        "   - Quantify everything possible\n"
        "3. **Section Reordering**: Put the most relevant experience sections first\n"
        "4. **Skills Section**: Reorder to prioritize JD-matching skills\n"
        "5. **Gap Analysis**: Identify requirements from the JD that the resume doesn't address\n"
        "6. **Match Score**: Rate the resume-JD match (1-100) before and after tailoring\n\n"
        "IMPORTANT: Do NOT invent or fabricate any experience. Only reframe existing experience."
    ),
    expected_output=(
        "The complete tailored resume with: new summary, rewritten bullets, reordered "
        "sections, optimized skills list, gap analysis, and before/after match score."
    ),
    agent=resume_tailor,
)

print(f"Task created: Resume Tailoring -> {task_resume_tailor.agent.role}")

### 5.3 Task 3 — Cover Letter

In [ ]:
task_cover_letter = Task(
    description=(
        "Write a compelling cover letter for this application.\n\n"
        "You have access to:\n"
        "- The JD analysis (from Task 1) with prioritized requirements and culture signals\n"
        "- The tailored resume (from Task 2) with optimized experience bullets\n\n"
        "Requirements:\n"
        "1. **Opening Hook**: Start with a specific, relevant accomplishment — NOT 'I was excited to see'\n"
        "2. **Bridge Section**: Connect the candidate's top 2-3 experiences directly to the role's "
        "   highest-priority requirements\n"
        "3. **Company Insight**: Demonstrate understanding of the company's mission, challenges, or "
        "   recent developments based on JD signals\n"
        "4. **Differentiation**: Articulate what makes this candidate uniquely valuable beyond "
        "   just meeting the requirements\n"
        "5. **Closing**: Confident, specific — reference a concrete next step\n"
        "6. **Length**: Under 300 words\n"
        "7. **Tone**: Professional but authentic — match the company's culture signals\n\n"
        "Do NOT simply restate the resume. The cover letter adds narrative, motivation, and context."
    ),
    expected_output=(
        "A complete cover letter under 300 words with: hook opening, experience-to-requirement "
        "bridges, company insight, differentiation statement, and confident close."
    ),
    agent=cover_letter_writer,
)

print(f"Task created: Cover Letter -> {task_cover_letter.agent.role}")

### 5.4 Task 4 — Interview Prep

In [ ]:
task_interview_prep = Task(
    description=(
        "Prepare a comprehensive interview preparation guide.\n\n"
        "You have access to ALL previous outputs: JD analysis, tailored resume, and cover letter.\n\n"
        "Produce:\n"
        "1. **Technical Questions** (5 questions):\n"
        "   - The likely question\n"
        "   - Why the interviewer asks this (what they're evaluating)\n"
        "   - A strong answer outline using the candidate's actual experience\n\n"
        "2. **Behavioral Questions** (5 questions):\n"
        "   - The likely question\n"
        "   - STAR-format answer: Situation, Task, Action, Result\n"
        "   - Each answer must use a REAL experience from the resume\n\n"
        "3. **Culture-Fit Questions** (3 questions):\n"
        "   - Based on the culture signals from the JD analysis\n"
        "   - Suggested responses that align with the company's values\n\n"
        "4. **Red-Flag Questions** (2-3 questions):\n"
        "   - Questions about gaps or weaknesses identified in the gap analysis\n"
        "   - Strategic responses that are honest but frame gaps positively\n\n"
        "5. **Questions to Ask the Interviewer** (5 questions):\n"
        "   - Smart, specific questions that demonstrate research and genuine interest\n\n"
        "6. **Interview Strategy**:\n"
        "   - Top 3 talking points to weave into every answer\n"
        "   - Key metrics/achievements to mention by name\n"
        "   - Body language and delivery tips for this company's culture\n"
    ),
    expected_output=(
        "A complete interview prep guide with: 5 technical questions with answers, "
        "5 behavioral questions with STAR answers, 3 culture-fit questions, 2-3 red-flag "
        "questions with strategic responses, 5 questions to ask, and an interview strategy."
    ),
    agent=interview_coach,
)

print(f"Task created: Interview Prep -> {task_interview_prep.agent.role}")

### Task Pipeline Visualization

In [ ]:
tasks = [task_jd_analysis, task_resume_tailor, task_cover_letter, task_interview_prep]
task_names = ["JD Analysis", "Resume Tailoring", "Cover Letter", "Interview Prep"]

print("TASK PIPELINE")
print("=" * 65)
for i, (name, task) in enumerate(zip(task_names, tasks), 1):
    status = "FINAL OUTPUT" if i == len(tasks) else f"feeds into Task {i+1}"
    print(f"  Task {i}: {name}")
    print(f"    Agent: {task.agent.role}")
    print(f"    Flow:  {status}")
    if i < len(tasks):
        print(f"    {'':8}|")
print("=" * 65)
print(f"\nContext accumulation: Task 4 sees ALL previous outputs (JD + Resume + Cover Letter)")

## 6. Assemble and Run the Crew

### How the Crew Coordinates Agents

The `Crew` orchestrator manages the assembly line:

| Step | What Happens |
|------|-------------|
| 1 | Crew sends the JD to the JD Analyzer → receives structured analysis |
| 2 | Crew sends (JD analysis) + resume to the Resume Tailor → receives tailored resume |
| 3 | Crew sends (JD analysis + tailored resume) to Cover Letter Writer → receives letter |
| 4 | Crew sends (ALL: analysis + resume + letter) to Interview Coach → receives prep guide |

Each agent operates independently — it only knows about its role and the accumulated context it receives.

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")

In [ ]:
# Launch the job application pipeline
print("=" * 70)
print(f"LAUNCHING JOB APPLICATION CREW — {datetime.now().strftime('%H:%M:%S')}")
print("Pipeline: JD Analysis -> Resume Tailor -> Cover Letter -> Interview Prep")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 7. Analyze Results

### 7.1 Final Output — Interview Prep Guide

In [ ]:
print("=" * 70)
print("FINAL OUTPUT — Interview Preparation Guide")
print("=" * 70)
print(result.raw)

### 7.2 Individual Agent Outputs

In [ ]:
for name, task in zip(task_names, tasks):
    print("\n" + "=" * 70)
    print(f"AGENT OUTPUT: {name}")
    print(f"Role: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2500:
            print(text[:2500])
            print(f"\n... [{len(text) - 2500} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 7.3 Pipeline Metrics

In [ ]:
print("APPLICATION PIPELINE METRICS")
print("=" * 60)
print(f"{'Stage':<20} {'Agent':<40} {'Output':>8}")
print("-" * 60)

total = 0
for name, task in zip(task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    print(f"{name:<20} {task.agent.role:<40} {length:>6,}")

print("-" * 60)
print(f"{'TOTAL':<60} {total:>6,}")

if hasattr(result, 'token_usage') and result.token_usage:
    print(f"\nTokens: {result.token_usage.get('total_tokens', 'N/A')}")
else:
    print("\nToken usage: not tracked (typical with local Ollama)")

## 8. Save Full Application Package

In [ ]:
# Save each deliverable separately
outputs = {}
for name, task in zip(task_names, tasks):
    if task.output:
        outputs[name] = task.output.raw

# Combined report
report_lines = [
    "# Job Application Package",
    f"**Generated:** {datetime.now().isoformat()}",
    f"**Position:** Senior Machine Learning Engineer — FinTech Startup",
    "---",
]
for name in task_names:
    if name in outputs:
        report_lines.append(f"\n## {name}\n")
        report_lines.append(outputs[name])
        report_lines.append("\n---")

report = "\n".join(report_lines)
with open("application_package.md", "w", encoding="utf-8") as f:
    f.write(report)

# Save individual files
if "Cover Letter" in outputs:
    with open("cover_letter.md", "w", encoding="utf-8") as f:
        f.write(outputs["Cover Letter"])

if "Interview Prep" in outputs:
    with open("interview_prep.md", "w", encoding="utf-8") as f:
        f.write(outputs["Interview Prep"])

print(f"Files saved:")
print(f"  application_package.md ({len(report):,} chars)")
if "Cover Letter" in outputs:
    print(f"  cover_letter.md ({len(outputs['Cover Letter']):,} chars)")
if "Interview Prep" in outputs:
    print(f"  interview_prep.md ({len(outputs['Interview Prep']):,} chars)")

## 9. Experiment: Different Job Description

Demonstrate pipeline reusability with a completely different role.

In [ ]:
# =====================================================
# SECOND JOB DESCRIPTION
# =====================================================
JOB_DESCRIPTION_2 = """
Staff Data Engineer — E-Commerce Platform (Public Company)

About the Role:
We're looking for a Staff Data Engineer to lead the design and implementation
of our next-generation data platform. You'll own the architecture decisions
that power analytics, ML pipelines, and real-time personalization for 100M+ users.

Requirements:
- 7+ years of data engineering experience
- Expert-level SQL and Python
- Deep experience with Spark (PySpark), Airflow, and dbt
- Designed and maintained data warehouses (Snowflake, Redshift, or BigQuery)
- Experience with real-time data pipelines (Kafka, Kinesis)
- Strong understanding of data modeling (star schema, data vault)
- Experience leading technical projects and mentoring junior engineers
- Infrastructure-as-code (Terraform, Pulumi)

Nice to Have:
- Experience with data mesh or data product architectures
- Knowledge of ML feature stores (Feast, Tecton)
- Experience with data quality frameworks (Great Expectations, dbt tests)
- Familiarity with privacy/compliance (GDPR, CCPA)

Culture: Collaborative, engineering-excellence focused. We do thorough code
reviews, invest in documentation, and believe in sustainable pace. No on-call
for data engineers.
"""

print("Second JD loaded")
print(f"Length: {len(JOB_DESCRIPTION_2):,} characters")

In [ ]:
# Build tasks for the second JD (same resume, different job)
task2_jd = Task(
    description=(
        f"Analyze this job description:\n\n{JOB_DESCRIPTION_2}\n\n"
        "Produce: must-haves (ranked), nice-to-haves, hidden requirements, "
        "ATS keywords, culture signals, seniority calibration, and red flags."
    ),
    expected_output="Structured JD analysis with 7 sections.",
    agent=jd_analyzer,
)

task2_resume = Task(
    description=(
        f"Tailor this resume for the role:\n\n{RESUME}\n\n"
        "Rewrite bullets, reorder sections, add ATS keywords, do NOT fabricate. "
        "Include a match score before/after."
    ),
    expected_output="Complete tailored resume with gap analysis and match score.",
    agent=resume_tailor,
)

task2_cover = Task(
    description=(
        "Write a cover letter under 300 words. Hook opening, bridge 2-3 key "
        "experiences to requirements, show company insight, differentiate, "
        "confident close. Do NOT restate the resume."
    ),
    expected_output="Cover letter under 300 words.",
    agent=cover_letter_writer,
)

task2_interview = Task(
    description=(
        "Prepare interview guide: 5 technical questions with answers, "
        "5 behavioral STAR-format questions, 3 culture-fit questions, "
        "red-flag questions, 5 questions to ask, and an interview strategy."
    ),
    expected_output="Complete interview prep with STAR answers and strategy.",
    agent=interview_coach,
)

tasks_2 = [task2_jd, task2_resume, task2_cover, task2_interview]
print(f"Second JD tasks created: {len(tasks_2)} tasks")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print("Position: Staff Data Engineer — E-Commerce")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print("INTERVIEW PREP — Staff Data Engineer")
print("=" * 70)
print(result2.raw)

## 10. Compare Both Applications

In [ ]:
print("APPLICATION COMPARISON")
print("=" * 70)
print(f"{'Stage':<20} {'ML Engineer (chars)':<22} {'Data Engineer (chars)':<22}")
print("-" * 70)

for name, t1, t2 in zip(task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{name:<20} {len1:<22,} {len2:<22,}")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 70)
print(f"{'TOTAL':<20} {total1:<22,} {total2:<22,}")

## 11. Role Design Deep Dive

### Anatomy of a Well-Designed Agent Role

Let's dissect the Resume Tailor agent as an example:

```python
Agent(
    # IDENTITY ANCHOR — Specific title, not generic
    role="Resume Optimization & ATS Specialist",
    
    # MISSION — Narrow, measurable objective
    goal="Tailor the resume to maximize match... without fabricating",
    
    # EXPERTISE CONTEXT — Shapes reasoning patterns
    backstory=(
        "You are a professional resume writer who has optimized 5,000+ resumes..."
        # ↑ Authority signal — the agent "believes" it's expert-level
        
        "You specialize in ATS optimization..."
        # ↑ Domain focus — prevents generic advice
        
        "Your approach: (1) Mirror keywords, (2) Reorder bullets..."
        # ↑ Methodology — gives the agent specific techniques to apply
        
        "CRITICAL: You NEVER invent experience."
        # ↑ Safety constraint — embedded in identity, not just instructions
    ),
)
```

### Role Design Patterns

| Pattern | Example | Effect |
|---------|---------|--------|
| **Authority signal** | "15 years of experience", "reviewed 10,000+" | Agent produces more confident, detailed output |
| **Methodology embedding** | "You follow the STAR method" | Agent applies the named framework consistently |
| **Output format** | "You always produce structured tables" | Output is predictable and parseable |
| **Safety constraint** | "You NEVER fabricate" | Prevents harmful behavior at the identity level |
| **Domain vocabulary** | "ATS", "keyword density", "XYZ formula" | Agent uses correct terminology and concepts |
| **Dual perspective** | "Both sides of the table" | Richer analysis from multiple viewpoints |

### Common Role Design Anti-Patterns

| Anti-Pattern | Why It Fails | Better Alternative |
|-------------|-------------|-------------------|
| `role="Assistant"` | No identity anchor; generic output | Use specific professional titles |
| `goal="Help the user"` | No direction; output wanders | State the specific deliverable |
| `backstory="You are helpful"` | No expertise context | Include years, methods, domain knowledge |
| Vague backstory, detailed task | Agent doesn't internalize expertise | Move methodology into backstory |
| `allow_delegation=True` always | Agents pass work around; no accountability | Default to False; enable only for managers |

### Testing Role Design

To test if your role design is effective, ask: **Would replacing this agent's backstory with "You are helpful" change the output quality?** If yes, the backstory is doing its job.

## 12. Advanced: Custom Role with Explicit Context

Demonstrate how to give an agent selective access to upstream outputs using the `context` parameter — useful when an agent needs specific inputs, not the full history.

In [ ]:
# Selective context example: Cover letter sees JD analysis + resume but NOT interview prep
# This is useful if you want to run cover letter and interview prep in parallel

task_ctx_jd = Task(
    description=f"Analyze this JD:\n{JOB_DESCRIPTION[:500]}...\nProduce ranked requirements and ATS keywords.",
    expected_output="JD analysis with ranked requirements.",
    agent=jd_analyzer,
)

task_ctx_resume = Task(
    description=f"Tailor this resume:\n{RESUME[:500]}...\nRewrite bullets, add keywords, match score.",
    expected_output="Tailored resume.",
    agent=resume_tailor,
    context=[task_ctx_jd],  # Receives JD analysis
)

# Both cover letter AND interview prep receive JD + resume (fan-out pattern)
task_ctx_cover = Task(
    description="Write a cover letter under 300 words using the JD analysis and tailored resume.",
    expected_output="Cover letter.",
    agent=cover_letter_writer,
    context=[task_ctx_jd, task_ctx_resume],  # Receives both, not cumulative
)

task_ctx_interview = Task(
    description="Prepare 5 technical + 5 behavioral STAR questions with answers.",
    expected_output="Interview prep guide.",
    agent=interview_coach,
    context=[task_ctx_jd, task_ctx_resume],  # Same inputs as cover letter
)

ctx_tasks = [task_ctx_jd, task_ctx_resume, task_ctx_cover, task_ctx_interview]

print("SELECTIVE CONTEXT HANDOFF")
print("=" * 60)
for i, task in enumerate(ctx_tasks, 1):
    ctx_names = [t.agent.role.split("&")[0].strip() for t in (task.context or [])]
    ctx_str = " + ".join(ctx_names) if ctx_names else "none (first task)"
    print(f"  Task {i}: {task.agent.role.split('&')[0].strip()}")
    print(f"    Receives: {ctx_str}")

print("\nNote: Tasks 3 and 4 receive the SAME context — they could run in parallel.")
print("Neither depends on the other's output.")

In [ ]:
# Run the selective-context crew
crew_ctx = Crew(
    agents=agents,
    tasks=ctx_tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching selective-context crew — {datetime.now().strftime('%H:%M:%S')}")
result_ctx = crew_ctx.kickoff()
print(f"\nSelective-context crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Show context flow results
print("SELECTIVE CONTEXT RESULTS")
print("=" * 60)
ctx_task_names = ["JD Analysis", "Resume Tailoring", "Cover Letter", "Interview Prep"]
for name, task in zip(ctx_task_names, ctx_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx_sources = [t.agent.role.split("&")[0].strip() for t in (task.context or [])]
    ctx_str = ", ".join(ctx_sources) if ctx_sources else "none"
    print(f"  {name:<20} {length:>6,} chars | Context from: {ctx_str}")

## 13. Key Takeaways

### What We Built
- A **4-agent pipeline** (JD Analyzer → Resume Tailor → Cover Letter Writer → Interview Coach)
- Ran it on **two different job descriptions** (ML Engineer + Data Engineer) with the same resume
- Demonstrated **selective context** with explicit `context` parameter

### Role Design Lessons (Summary)

1. **Role = Identity** — Use specific professional titles, not generic labels
2. **Goal = Mission** — Narrow scope produces better output than "be helpful"
3. **Backstory = Expertise** — Include years, methodologies, domain vocabulary, and quality standards
4. **Constraints in backstory** — Safety rules ("never fabricate") are more effective as identity than instructions
5. **Methodology embedding** — Name specific frameworks (STAR, XYZ, PAS) the agent should apply
6. **Dual perspective** — Agents that understand both sides (interviewer + candidate) produce richer output
7. **Output format in persona** — Telling the agent it "always produces structured tables" improves consistency

### Production Tips
- Add web search tools to the JD Analyzer for real company/industry research
- Use `output_pydantic` to enforce structured schemas (especially for the JD analysis)
- Enable `memory=True` to learn from multiple applications over time
- Consider adding a **Networking Agent** that identifies mutual connections and referral paths
- Add a **Salary Negotiator** agent that researches market rates and prepares negotiation talking points
- Use callbacks to log each agent's output for A/B testing different role designs